# Task 4: Interactive Full-Stack AI Chatbot

**Project:** DocQA Chat, a streaming web app for your own documents
**Language and stack:** Python 3.12, Streamlit, Retrieval-Augmented Generation (reusing the Task 2 pipeline), OpenRouter (`openai/gpt-oss-20b`), pytest with Streamlit `AppTest`, ruff, uv

This notebook walks through my Task 4 project from top to bottom. It shows the source of the web app, explains how streaming, memory and per-session storage work, and can start the real Streamlit app from inside the notebook. It does not copy the code. It reads the project files, so what you see here is the code the tests check.

## Contents

1. Objective and key concepts
2. Setup
3. What the app does
4. Architecture
5. Source code tour: the web app (`app.py`, `ui.py`, `workspace.py`, `uploads.py`, `theme.py`, `style.css`, theme config)
6. Source code tour: the streaming pieces in the Task 2 pipeline
7. Each visitor gets a private store
8. Run the app
9. Screenshots
10. Tests
11. Design decisions, limitations, skills

## 1. Objective and key concepts

**Objective:** build an interactive chatbot with a web interface. DocQA Chat is a Streamlit app on top of the Task 2 RAG pipeline. You upload your own documents, ask questions in plain words, and watch the answer stream in token by token. Every answer comes only from your documents, with numbered sources. When the documents do not contain the answer, the app says so.

**Key concepts**

- Streamlit user interface: sidebar, chat, source chips, custom theme
- Streaming responses: the answer appears as the model writes it, then is replaced by the final text with clean citations
- Conversation memory: the last 6 exchanges, updated only after an answer has streamed to the end
- Session state management: chat history, memory, document store and upload results live in `st.session_state`
- RAG integration: reuses the Task 2 pipeline (`DocQAPipeline.chat_stream`)
- Per-session document store: each visitor gets a private temporary vector store
- Upload safety: file type, size and count limits, path-safe file names, zip files read in memory, HTML-escaped source chips
- Headless UI testing with Streamlit's `AppTest`

## 2. Setup

In [1]:
import json
import os
import subprocess
import sys
import tempfile
import time
import urllib.request
from pathlib import Path

SKIP_FOLDERS = {".venv", "venv", "node_modules", ".git", "__pycache__", ".ipynb_checkpoints"}


def find_folder(marker: str, start: Path | None = None) -> Path | None:
    """Find the folder that contains `marker` (a relative path such as docqa/cli.py).

    Looks at the start folder and its parents first, then up to three levels below it,
    so it works from the repository root or from inside a project folder.
    """
    here = (start or Path.cwd()).resolve()
    parts = marker.split("/")

    for folder in [here, *here.parents]:
        if folder.joinpath(*parts).exists():
            return folder

    for current, subfolders, _ in os.walk(here):
        depth = len(Path(current).relative_to(here).parts)
        subfolders[:] = (
            [d for d in subfolders if d not in SKIP_FOLDERS and not d.startswith(".")]
            if depth < 3 else []
        )
        if Path(current).joinpath(*parts).exists():
            return Path(current)
    return None


STARTED_IN = Path.cwd().resolve()
PROJECT = find_folder("docqa_web/ui.py")
if PROJECT is None:
    raise FileNotFoundError(
        "Could not find the Streamlit project (a folder containing docqa_web/ui.py). "
        "Start Jupyter from the repository root or from the Streamlit project folder."
    )

# The app imports the Task 2 package (docqa), so find that folder too.
TASK2 = find_folder("docqa/cli.py", start=PROJECT.parent)

os.chdir(PROJECT)
for folder in (PROJECT, TASK2):
    if folder is not None and str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

print("Notebook started in:", STARTED_IN.name)
print("Streamlit project folder:", PROJECT.name)
print("Task 2 pipeline folder:", TASK2.name if TASK2 else "NOT FOUND")
print("Python:", sys.version.split()[0])

Notebook started in: Datheon_Generative_AI_Internship
Streamlit project folder: Task4_DocQA_Streamlit
Task 2 pipeline folder: Task2_DocQA_RAG
Python: 3.12.12


In [2]:
# Check that an API key is configured, without printing any secret.
# The app looks for a .env file in its own folder or any parent folder.
env_file = next((f / ".env" for f in [PROJECT, *PROJECT.parents] if (f / ".env").exists()), None)
print(".env file found:", env_file is not None)
if env_file:
    names = set()
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            names.add(line.split("=", 1)[0].strip())
    print("OPENROUTER_API_KEY present:", "OPENROUTER_API_KEY" in names)

.env file found: True
OPENROUTER_API_KEY present: True


This notebook can live in the repository root or inside the Streamlit project folder. The first cell finds both the app folder and the Task 2 folder by itself, so the paths and `PYTHONPATH` are set up correctly. Start Jupyter from the repository root with `uv run --with jupyter jupyter lab`.

## 3. What the app does

- **Bring your own documents.** Drop many `.md`, `.txt` and `.pdf` files at once, drop a `.zip` (its documents are added), or type a folder path to load everything in it and its sub-folders.
- **Streaming answers.** The reply appears as the model writes it. When it ends, the display is replaced by the final text, with `[n]` markers cleaned up, and a collapsible **Sources** list shows the file, the page or section, and a snippet for each citation.
- **Conversation memory.** The last 6 exchanges are remembered, so a follow-up such as "what about the second one?" is understood. The memory is updated only after an answer has streamed to the end, so a stopped or failed stream never leaves half an answer behind.
- **Clear session state.** Chat history, memory, the document store and the upload results all live in `st.session_state`. **Clear chat** forgets the conversation. **Remove all documents** also deletes the session's store.
- **Honest errors.** A file that cannot be read is reported by name and does not stop the others. A network failure mid-answer keeps the partial text and shows the error. A missing API key is shown as a message, not a crash.
- **A look of its own.** Colour theme, gradient header, chat bubbles, source chips and a "how it works" empty state.

## 4. Architecture

```
 browser  --upload / zip / folder path-->  sidebar (ui.py)
                                              |
                                   uploads.py: which files are new?
                                              |
                                   workspace.py: check, save, ingest one file at a time
                                              |                  (temp folder + own Chroma store)
 question --> chat_input (ui.py)              v
      |                                  DocQAPipeline (Task 2)
      +--> pipeline.chat_stream(question, memory)
              1. rewrite a follow-up as a standalone question (only with history)
              2. retrieve the top passages, apply the relevance threshold
              3. stream the answer from the LLM, piece by piece      --> placeholder.markdown(...)
              4. at the end: final answer + citations, memory updated, telemetry written
```

```
04_DocQA_Streamlit/
  app.py                   entry point (three lines)
  .streamlit/config.toml   colour theme
  docqa_web/
    ui.py                  the screen: sidebar, chat, session state
    workspace.py           per-session temp store, uploads, folders, zips (no Streamlit)
    uploads.py             which uploaded files are new, zip routing (no Streamlit)
    theme.py, style.css    look and feel
  tests/                   offline tests
```

## 5. Source code tour: the web app

The helper below prints a project file as it is on disk. It reads the file instead of importing it, so showing the source never starts the app.

In [3]:
def show_file(relative_path: str, base: Path | None = None) -> None:
    path = (base or PROJECT) / relative_path
    if not path.exists():
        print(f"File not found: {relative_path}")
        return
    text = path.read_text(encoding="utf-8")
    print(f"# {relative_path}  ({len(text.splitlines())} lines)\n")
    print(text)

### 5.1 Entry point (`app.py`)

Three lines. All the logic lives in the `docqa_web` package so it can be tested.

In [4]:
show_file("app.py")

# app.py  (11 lines)

"""Entry point for the DocQA chat web app.

Run from this folder, so Streamlit finds `.streamlit/config.toml` (the colour theme):

    cd 04_DocQA_Streamlit
    PYTHONPATH=../02_DocQA_RAG uv run streamlit run app.py
"""

from docqa_web.ui import render

render()



### 5.2 Theme configuration (`.streamlit/config.toml`)

Light theme with violet as the main colour, and the Deploy button hidden. Streamlit reads this file only when the app is started from the project folder.

In [5]:
show_file(".streamlit/config.toml")

# .streamlit/config.toml  (16 lines)

# Colour theme and tidy-ups for the DocQA chat app.
# Streamlit reads this file from the folder it is started in, so run the app from here.

[theme]
base = "light"
primaryColor = "#6C5CE7"
backgroundColor = "#FBFAFF"
secondaryBackgroundColor = "#F1EEFF"
textColor = "#1F1B3A"
font = "sans serif"

[client]
toolbarMode = "minimal"

[browser]
gatherUsageStats = false



### 5.3 The screen (`docqa_web/ui.py`)

The sidebar, the chat, the streaming display and all session state in `st.session_state`.

In [6]:
show_file("docqa_web/ui.py")

# docqa_web/ui.py  (206 lines)

"""The Streamlit screen: documents in the sidebar, a streaming chat in the main area.

All per-visitor state lives in `st.session_state`, created once by `_init_state`. Streamlit
re-runs this script on every click, so nothing expensive is built outside session state.
"""

from dataclasses import dataclass

import streamlit as st
from docqa.answer import Citation
from docqa.config import load_settings
from docqa.conversation import ConversationMemory
from docqa.errors import ConfigError, DocQAError
from docqa.loader import SUPPORTED_SUFFIXES
from docqa.logging_setup import configure_logging
from dotenv import find_dotenv, load_dotenv
from streamlit.delta_generator import DeltaGenerator

from docqa_web.theme import (
    banner_html,
    load_css,
    ready_pill_html,
    source_chip_html,
    steps_html,
    style_tag,
)
from docqa_web.uploads import ZIP_SUFFIX, ingest_new_uploads
from docqa_web.workspace import DocumentWorkspace, FileResult

PAGE_TITLE =

### 5.4 Per-session workspace (`docqa_web/workspace.py`)

Gives each browser session a temporary folder and its own vector store. Checks, saves and ingests each upload one file at a time, and handles folders and zip files. It contains no Streamlit code, so it is tested directly.

In [7]:
show_file("docqa_web/workspace.py")

# docqa_web/workspace.py  (252 lines)

"""A private, temporary document workspace for one browser session.

Each workspace owns a temporary folder holding the uploaded files and a vector store of its
own, so nothing a visitor uploads touches the project's main store, and everything is deleted
when the workspace is reset or closed. Files arrive one at a time, from a folder on this
computer, or inside a zip. This module has no Streamlit imports.
"""

import io
import logging
import shutil
import tempfile
import weakref
import zipfile
import zlib
from collections import Counter
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path, PurePosixPath

from docqa.config import Settings
from docqa.errors import DocQAError
from docqa.loader import SUPPORTED_SUFFIXES
from docqa.pipeline import DocQAPipeline, build_pipeline

logger = logging.getLogger(__name__)

BYTES_PER_MB = 1024 * 1024
SESSION_PREFIX = "docqa-session-"
# One folder or zip adds at most th

### 5.5 Upload helpers (`docqa_web/uploads.py`)

Decides which uploaded files are new and routes zip files. It contains no Streamlit code.

In [8]:
show_file("docqa_web/uploads.py")

# docqa_web/uploads.py  (40 lines)

"""Turn the files a browser sent into workspace calls. No Streamlit imports."""

from collections.abc import Iterable
from typing import Protocol

from docqa_web.workspace import DocumentWorkspace, FileResult

ZIP_SUFFIX = ".zip"


class Upload(Protocol):
    """What the app needs from an uploaded file (Streamlit's UploadedFile has all of it)."""

    file_id: str
    name: str

    def getvalue(self) -> bytes:
        """The file's content."""
        ...


def ingest_new_uploads(
    workspace: DocumentWorkspace, uploads: Iterable[Upload], seen: set[str]
) -> list[FileResult]:
    """Ingest the uploads not handled before; a zip gives one result per file inside it.

    Streamlit re-runs the script on every click and the uploader keeps returning the same
    files, so `seen` (file ids) stops a file being read twice. A file that failed is marked
    as seen too, so it is not retried, and re-reported, on every later click.
    """
    results: list[Fi

### 5.6 Theme pieces (`docqa_web/theme.py`)

The small HTML pieces (banner, cards, source chips, status pill), with HTML escaping and no Streamlit calls.

In [9]:
show_file("docqa_web/theme.py")

# docqa_web/theme.py  (64 lines)

"""Look and feel: the stylesheet and the small HTML pieces the screen uses.

No Streamlit calls here, so it can be tested on its own. Anything that comes from a user
(file names, document locations) is HTML-escaped before it is put into a page.
"""

import html
from pathlib import Path

from docqa.answer import Citation

STYLE_PATH = Path(__file__).with_name("style.css")
TITLE = "\U0001f4c4 DocQA Chat"
TAGLINE = "Ask questions about your own documents. Every answer comes with its sources."
STEPS = (
    ("1 · Upload", "Drop files or a zip, or point to a folder on this computer."),
    ("2 · Ask", "Type a question in plain words. Follow-up questions work too."),
    ("3 · Check", "Each answer lists the file and page it came from."),
)


def load_css(path: Path = STYLE_PATH) -> str:
    """The stylesheet text."""
    return path.read_text(encoding="utf-8")


def style_tag(css: str) -> str:
    """The stylesheet wrapped so Streamlit can inject it into the

### 5.7 Stylesheet (`docqa_web/style.css`)

Accent colours only: gradient banner, chat bubbles, source chips and a narrow-screen layout. It sets no page or text colours, so it works on top of any base theme.

In [10]:
show_file("docqa_web/style.css")

# docqa_web/style.css  (148 lines)

/* DocQA Chat look and feel.
   Accents only (no page or text colours), so it sits on top of any Streamlit base theme. */

:root {
  --dq-violet: #6c5ce7;
  --dq-violet-soft: #8e6cf0;
  --dq-teal: #00b8a9;
  --dq-mint: #2ed8a7;
  --dq-coral: #ff7675;
  --dq-sun: #fdcb6e;
}

.block-container {
  padding-top: 1.6rem;
  max-width: 52rem;
}

/* ---------- header banner ---------- */
.dq-banner {
  background: linear-gradient(120deg, var(--dq-violet) 0%, var(--dq-violet-soft) 45%, var(--dq-coral) 100%);
  color: #ffffff;
  border-radius: 20px;
  padding: 1.5rem 1.7rem;
  margin-bottom: 1.3rem;
  box-shadow: 0 12px 32px rgba(108, 92, 231, 0.30);
}
.dq-title {
  font-size: 2rem;
  font-weight: 800;
  letter-spacing: -0.015em;
  line-height: 1.15;
}
.dq-tagline {
  margin-top: 0.4rem;
  font-size: 1.02rem;
  opacity: 0.94;
}

/* ---------- "how it works" cards ---------- */
.dq-steps {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(11rem, 

## 6. Source code tour: the streaming pieces in the Task 2 pipeline

The streaming itself lives in the Task 2 package as two additive pieces, so the command line and the web app share one pipeline:

- `AnswerGenerator.answer_stream` returns an `AnswerStream`. Iterate it for text pieces, then read `.answer` for the finished `Answer` (citations, refusal flag, token counts). Opening the stream is retried on transient errors. A failure part-way raises `LLMCallError` and leaves no answer.
- `DocQAPipeline.chat_stream` returns a `ChatStream` that wraps it and writes memory and telemetry only when the stream has been read to the end.

`answer()` and `chat()` were refactored to share their helpers with the streaming path and behave as before.

In [11]:
import importlib
import inspect


def show_object(object_name: str, member: str | None = None) -> None:
    """Print the source of a class (or one of its methods) from the Task 2 package."""
    for module_name in ("docqa.answer", "docqa.pipeline", "docqa.conversation"):
        try:
            module = importlib.import_module(module_name)
        except Exception as error:
            print(f"Could not import {module_name}: {error}")
            continue
        target = getattr(module, object_name, None)
        if target is None:
            continue
        if member:
            target = getattr(target, member, None)
            if target is None:
                continue
        print(f"# {module_name}.{object_name}{'.' + member if member else ''}\n")
        print(inspect.getsource(target))
        return
    print(f"Could not find {object_name}{'.' + member if member else ''} in the Task 2 package.")

### `AnswerStream`

In [12]:
show_object("AnswerStream")

# docqa.answer.AnswerStream

class AnswerStream:
    """An answer arriving in pieces. Iterate once; `answer` is ready when iteration ends.

    The streamed text is raw: markers are normalized and a refusal is replaced only in the
    final `answer`, so a front end should swap its display for `answer.text` at the end.
    If the stream fails part-way, iteration raises LLMCallError and `answer` stays unavailable.
    """

    def __init__(self, pieces: Iterable[str], finish: Callable[[str], Answer]) -> None:
        self._pieces = pieces
        self._finish = finish
        self._parts: list[str] = []
        self._answer: Answer | None = None

    def __iter__(self) -> Iterator[str]:
        try:
            for piece in self._pieces:
                self._parts.append(piece)
                yield piece
        except openai.OpenAIError as exc:
            raise LLMCallError(f"LLM stream failed: {exc}") from exc
        self._answer = self._finish("".join(self._parts))

    @property


### `AnswerGenerator.answer_stream`

In [13]:
show_object("AnswerGenerator", "answer_stream")

# docqa.answer.AnswerGenerator.answer_stream

    def answer_stream(
        self,
        question: str,
        retrieved: Sequence[RetrievedChunk],
        history: Sequence[Turn] = (),
    ) -> AnswerStream:
        """Like `answer`, but the text arrives in pieces; read it to the end for the Answer."""
        question = validate_question(question, self._max_question_chars)
        if not retrieved:
            logger.info("no relevant chunks; refusing without an LLM call")
            refusal = Answer(text=REFUSAL_MESSAGE, citations=(), refused=True)
            return AnswerStream([REFUSAL_MESSAGE], lambda _text: refusal)
        citations = _build_citations(retrieved)
        messages = self._messages(question, citations, retrieved, history)
        usage: dict[str, int | None] = _tokens(None)
        response = self._complete(messages, stream=True)

        def finish(text: str) -> Answer:
            if not text.strip():
                raise LLMCallError("the model returned a

### `ChatStream`

In [14]:
show_object("ChatStream")

# docqa.pipeline.ChatStream

class ChatStream:
    """A chat answer arriving in pieces; memory and telemetry are updated when it ends.

    Iterate once. If the reader stops early or the stream fails, nothing is recorded.
    """

    def __init__(self, stream: AnswerStream, on_complete: Callable[[Answer], None]) -> None:
        self._stream = stream
        self._on_complete = on_complete

    def __iter__(self) -> Iterator[str]:
        yield from self._stream
        self._on_complete(self._stream.answer)

    @property
    def answer(self) -> Answer:
        """The finished answer; only available after the stream has been read to the end."""
        return self._stream.answer



### `DocQAPipeline.chat_stream`

In [15]:
show_object("DocQAPipeline", "chat_stream")

# docqa.pipeline.DocQAPipeline.chat_stream

    def chat_stream(self, question: str, memory: ConversationMemory) -> ChatStream:
        """Like `chat`, but the answer streams in pieces.

        The follow-up is rewritten and the passages are retrieved before this returns, so
        those errors surface here. The exchange is added to `memory` and the telemetry record
        is written only once the stream has been read to the end.
        """
        question = validate_question(question, self._max_question_chars)
        history = memory.turns
        started = self._clock()
        search_text, retrieved = self._search(question, history)
        retrieved_at = self._clock()
        stream = self._generator.answer_stream(search_text, retrieved, history)

        def on_complete(answer: Answer) -> None:
            self._record(question, len(retrieved), answer, started, retrieved_at)
            memory.add(question, answer.text)

        return ChatStream(stream, on_complete)



## 7. Each visitor gets a private store

`DocumentWorkspace` (`docqa_web/workspace.py`) gives every browser session a temporary folder and its own vector store inside it, so nothing a visitor uploads touches the project's main store (`02_DocQA_RAG/data/chroma`). The folder is deleted on **Remove all documents** and when the session ends. Usage telemetry is not recorded for these temporary stores.

Each upload is checked and ingested on its own, and gets its own result:

| Check | Result |
|---|---|
| Not `.md`, `.txt`, `.pdf` (or `.zip`) | Reported as unsupported, nothing saved |
| Empty, or larger than `DOCQA_MAX_FILE_MB` (20 MB by default) | Reported, nothing saved |
| Cannot be read or embedded | Reported by name, removed, not retried |
| Same name uploaded again | Replaces the earlier version |
| Two files with the same name in different folders | Both load, as `one_a.md` and `two_a.md` |

A folder or zip adds at most 200 files or 200 MB. The app says how many were skipped.

**Safety.** File names are reduced to their last part, so `../../x.md` cannot write outside the session folder. Zip entries are read in memory and never extracted to a path. Hidden files and folders, macOS `__MACOSX` entries and password-protected zip entries are skipped or reported. File names and locations are HTML-escaped before they are shown in a source chip.

## 8. Run the app

The app is a web server, so the normal way to run it is in a terminal, **from the project folder** so Streamlit finds `.streamlit/config.toml`:

```bash
cd 04_DocQA_Streamlit
PYTHONPATH=../02_DocQA_RAG uv run streamlit run app.py
```

`PYTHONPATH` points at the Task 2 package (`docqa`), which the app imports. Streamlit prints a local address, usually http://localhost:8501.

The two cells below start and stop the same app from inside this notebook and show it in a frame. They are switched off by default, so "Run All Cells" does not leave a server running. Set `START_APP = True` to use them. The app needs the API key in `.env` to answer questions.

In [21]:
START_APP = True
PORT = 8501
app_process = None

if START_APP:
    if TASK2 is None:
        raise RuntimeError("The Task 2 folder (docqa/cli.py) was not found, so the app cannot import docqa.")
    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join(filter(None, [str(TASK2), env.get("PYTHONPATH")]))
    log_path = Path(tempfile.gettempdir()) / "docqa_streamlit_notebook.log"
    log_file = open(log_path, "w", encoding="utf-8")
    app_process = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "app.py",
         "--server.headless", "true", "--server.port", str(PORT),
         "--browser.gatherUsageStats", "false"],
        cwd=PROJECT, env=env, stdout=log_file, stderr=subprocess.STDOUT,
    )
    healthy = False
    for _ in range(60):
        try:
            with urllib.request.urlopen(f"http://localhost:{PORT}/_stcore/health", timeout=2) as response:
                healthy = response.read().decode().strip() == "ok"
        except Exception:
            time.sleep(1)
        if healthy:
            break
    print("App is running at", f"http://localhost:{PORT}" if healthy else "(did not start; see the log below)")
    if not healthy:
        print(log_path.read_text(encoding="utf-8")[-1500:])
else:
    print("START_APP is False. Start the app in a terminal (see above) or set START_APP = True.")

App is running at http://localhost:8501


In [17]:
from IPython.display import IFrame, display

if app_process is not None and app_process.poll() is None:
    display(IFrame(src=f"http://localhost:{PORT}", width="100%", height=720))

In [18]:
# Stop the app when you are finished.
if app_process is not None and app_process.poll() is None:
    app_process.terminate()
    app_process.wait(timeout=15)
    print("App stopped.")
else:
    print("No app is running from this notebook.")

No app is running from this notebook.


## 9. Screenshots

If the project folder has a `screenshots/` folder with `.png` files, they are shown below. Add a few (the empty state, a streamed answer with sources, a refusal) to document how the app looks, since the visual appearance is checked by eye and not by the automated tests.

In [19]:
from IPython.display import Image, display

shots = sorted((PROJECT / "screenshots").glob("*.png")) if (PROJECT / "screenshots").exists() else []
if shots:
    for shot in shots:
        print(shot.name)
        display(Image(filename=str(shot), width=900))
else:
    print("No screenshots yet. Add .png files to a screenshots/ folder in the project.")

No screenshots yet. Add .png files to a screenshots/ folder in the project.


## 10. Tests

In [20]:
test_env = os.environ.copy()
if TASK2 is not None:
    test_env["PYTHONPATH"] = os.pathsep.join(filter(None, [str(TASK2), test_env.get("PYTHONPATH")]))

tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    capture_output=True, text=True, cwd=PROJECT, env=test_env,
)
print(tests.stdout[-1500:])
print(tests.stderr[-500:])

.......................................................................  [100%]
71 passed in 4.58s




The tests need no network and no API key. They cover:

- the streaming generator and pipeline: pieces in order, refusal, mid-stream failure, retry on opening, and memory and telemetry written only at the end
- the workspace: every check in the table in section 7, isolation from the main store, cleanup, folders, zips, limits and a password-protected entry
- the upload and theme helpers, including HTML escaping
- the screen itself, headlessly with Streamlit's `AppTest` and a fake workspace placed in `st.session_state`: the empty state and cards, the status pill, a streamed answer with sources, the hand-off of the session memory, a refusal, a failure part-way through an answer, Clear chat, Remove all documents, the folder box, a failed file listed by name, and the start-up messages for a missing API key and an invalid setting

These tests found a real bug (Clear chat stayed disabled after the first answer), which is fixed. How the page looks is not tested. It was checked by eye in the browser.

## 11. Design decisions, limitations, skills

**Design decisions**

- **A separate folder that imports Task 2.** Task 2 stays exactly as graded, and the web app is a thin front end. The only changes to Task 2 are additive (the streaming methods), with its tests.
- **Logic out of the screen.** Uploads, workspace and theme pieces are plain Python so they can be tested without a browser. The screen keeps all of its state in `st.session_state`.
- **One store per session, one file at a time.** It isolates visitors, gives a clear result per file, and means a bad file is removed instead of being retried on every click.
- **Accent-only styling.** No page or text colours in the CSS, so a different base theme cannot make text unreadable.
- **LangChain.** As in Task 2, LangChain is used only for the text splitter. Retrieval, prompting and streaming use the OpenAI-compatible client directly, which keeps the streaming and retry behaviour explicit and testable.

**Limitations**

- The chat and the uploaded documents belong to one browser session. Refreshing the page or closing the tab starts again, and the documents have to be uploaded and embedded again.
- The folder box reads whatever path it is given on the machine running the app. It is meant for running locally. Do not expose it on a public server.
- While an answer streams, the text is shown as the model wrote it. `[n]` markers are normalised, and a refusal is replaced by the fixed refusal sentence, only when it ends.
- The token counts of the rewrite call for follow-ups are not included in telemetry, and telemetry is off for the temporary stores.
- The limits of the Task 2 pipeline still apply: questions that need counting across many documents are refused, long synthesis answers are weak, and the relevance threshold was tuned on one corpus.
- There is no login and no sharing. It is a single-user local app.
- Some of the stylesheet targets Streamlit's own page structure (the chat bubbles). A future Streamlit release could change that. The app would keep working and lose only those colours.

**Skills shown in this project:** Python, Streamlit, streaming LLM responses, Retrieval-Augmented Generation (RAG), session state management, conversation memory, file upload handling, input validation and path-traversal protection, zip handling, HTML escaping, custom CSS theming, headless UI testing with Streamlit `AppTest`, separation of logic and UI, pytest, ruff, uv.